In [3]:
import pandas as pd

df = pd.read_csv("AI_Human.csv")

# Check the column names first
print(df.columns)

# First 10 human-written rows
human = df[df["generated"] == 0]
print(human["text"].head(5).to_string())

# First 10 AI-generated rows
ai = df[df["generated"] == 1]
print(ai["text"].head(5).to_string())

Index(['text', 'generated'], dtype='str')
0    Cars. Cars have been around since they became ...
1    Transportation is a large necessity in most co...
2    "America's love affair with it's vehicles seem...
3    How often do you ride in a car? Do you drive a...
4    Cars are a wonderful thing. They are perhaps o...
704     This essay will analyze, discuss and prove one...
740     I strongly believe that the Electoral College ...
1262    Limiting car use causes pollution, increases c...
1378    Car-free cities have become a subject of incre...
1379    Car Free Cities  Car-free cities, a concept ga...


In [4]:
# Looking at average sentence lengths
import re

def avg_sentence_length(text):
    # Split on . ! ? and drop empty pieces
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    if not sentences:
        return 0
    return sum(len(s.split()) for s in sentences) / len(sentences)

df["avg_sentence_len"] = df["text"].apply(avg_sentence_length)

sentence_result = df.groupby("generated")["avg_sentence_len"].mean()
sentence_result.index = ["Human", "AI"]
print(sentence_result)

Human    21.330323
AI       19.077838
Name: avg_sentence_len, dtype: float64


In [6]:
# Looking at number of each main type of punctuations
marks = {
    ",": "commas",
    ".": "periods",
    ";": "semicolons",
    ":": "colons",
    "?": "question marks",
    "!": "exclamation marks",
    '"': "quotes",
    "'": "apostrophes",
}

for mark, name in marks.items():
    # Average count per essay
    df[name] = df["text"].apply(lambda t: t.count(mark))
    # Count per 1,000 characters, so longer essays don't skew the result
    df[name + "_per_1000"] = df[name] / df["text"].str.len() * 1000

# Average count per essay
counts = df.groupby("generated")[list(marks.values())].mean()
counts.index = ["Human", "AI"]
print(counts.T)

# Per 1,000 characters (fairer comparison)
rates = df.groupby("generated")[[n + "_per_1000" for n in marks.values()]].mean()
rates.index = ["Human", "AI"]
print(rates.T)

                       Human         AI
commas             15.904054  20.590940
periods            21.043424  17.752367
semicolons          0.240163   0.068359
colons              0.209982   0.503649
question marks      0.809746   0.333502
exclamation marks   0.279875   0.416875
quotes              3.162111   0.812801
apostrophes         5.404036   3.598557
                               Human        AI
commas_per_1000             6.454186  9.708227
periods_per_1000            9.178817  8.533570
semicolons_per_1000         0.090037  0.034961
colons_per_1000             0.083323  0.197669
question marks_per_1000     0.357483  0.189549
exclamation marks_per_1000  0.134099  0.239966
quotes_per_1000             1.312107  0.381042
apostrophes_per_1000        2.415044  1.842896


In [13]:
import pandas as pd
import nltk
from collections import Counter

import ssl
import nltk

# Work around the macOS python.org SSL certificate problem
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print(nltk.download("punkt_tab"))
print(nltk.download("averaged_perceptron_tagger_eng"))

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

# Rebuild the 100-per-group sample
df = pd.read_csv("AI_Human.csv")
small = df.groupby("generated").head(100).copy()

def noun_stats(text):
    words = nltk.word_tokenize(text)
    tags = nltk.pos_tag(words)
    nouns = [w.lower() for w, tag in tags if tag.startswith("NN")]
    return pd.Series({"noun_count": len(nouns),
                      "nouns_per_100_words": len(nouns) / max(len(words), 1) * 100})

small[["noun_count", "nouns_per_100_words"]] = small["text"].apply(noun_stats)

noun_result = small.groupby("generated")[["noun_count", "nouns_per_100_words"]].mean()
noun_result = noun_result.rename(index={0: "Human", 1: "AI"})
print(noun_result)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mayahkg/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/mayahkg/nltk_data...


True


[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mayahkg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/mayahkg/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True
           noun_count  nouns_per_100_words
generated                                 
Human          169.07            24.571289
AI             162.18            30.310715
